<a href="https://colab.research.google.com/github/ishita452/GDSC_Drug_Ranking_ML/blob/main/Drug_sensitivity_in_cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score,KFold,GroupKFold, RandomizedSearchCV, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn. compose import ColumnTransformer
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from scipy.stats import spearmanr

**LOADING ALL THE DATASETS**

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
gdsc=pd.read_csv("GDSC_DATASET.csv") #main dataset
gdsc

In [ ]:
gdsc2=pd.read_csv("GDSC2-dataset.csv") #drug response data
gdsc2 #different number of rows in gdsc and gdsc2

In [ ]:
plt.hist(gdsc["LN_IC50"].dropna(), bins=50)
plt.xlabel("LN_IC50")
plt.ylabel("Frequency")
plt.title("Distribution of LN_IC50")
plt.show()

In [ ]:
cell_lines=pd.read_excel("Cell_Lines_Details.xlsx") #cell line details
cell_lines

In [ ]:
cell_lines = cell_lines.rename(columns={'COSMIC identifier': 'COSMIC_ID', 'Sample Name': 'CELL_LINE_NAME' ,
                                                'GDSC\nTissue descriptor 1':'GDSC Tissue descriptor 1',
                                                'GDSC\nTissue\ndescriptor 2':'GDSC Tissue descriptor 2',
                                                'Cancer Type\n(matching TCGA label)':'Cancer Type (matching TCGA label)',
                                                'Microsatellite \ninstability Status (MSI)':'MSI',
                                                'Whole Exome Sequencing (WES)': 'WES',
                                                'Copy Number Alterations (CNA)': 'CNA'
                                               })
cell_lines

In [ ]:
compounds=pd.read_csv("Compounds-annotation.csv") #drug information
compounds

**EXPLORATORY DATA ANALYSIS**

In [ ]:
print("gdsc",gdsc.shape)
print("gdsc2",gdsc2.shape)
print("cell lines",cell_lines.shape)
print("compounds",compounds.shape)


In [ ]:
print("gdsc",list(gdsc.columns))
print("gdsc2",list(gdsc2.columns))
print("cell lines",list(cell_lines.columns))
print("compounds",list(compounds.columns))


In [ ]:
print("gdsc",gdsc.info())
print("gdsc2",gdsc2.info())
print("cell lines",cell_lines.info())
print("compounds",compounds.info())

In [ ]:
print("gdsc",gdsc.count())
print("gdsc2",gdsc2.count())
print("cell lines",cell_lines.count())
print("compounds",compounds.count())

**MERGING DATASETS**

In [ ]:
merge_data=pd.merge(gdsc2,cell_lines,on=["COSMIC_ID","CELL_LINE_NAME"],how="left")
merge_data

In [ ]:
main_data=pd.merge(merge_data,compounds,on=["DRUG_ID","DRUG_NAME"],how="left")
main_data #final merged dataset with all columns

In [ ]:
print(list(main_data.columns))

In [ ]:
pd.crosstab(main_data['TCGA_DESC'], main_data['Cancer Type (matching TCGA label)'])

In [ ]:
#removing columns that could cause leakage and are not essential features
final_data=main_data.drop(columns=["DATASET", "NLME_RESULT_ID", "NLME_CURVE_ID", "COSMIC_ID",
                                    "SANGER_MODEL_ID","SYNONYMS","SCREENING_SITE","Screen Medium","PUTATIVE_TARGET",
                                   "PATHWAY_NAME","COMPANY_ID","WEBRELEASE","AUC","RMSE","Z_SCORE",
                                   "WES","CNA","Methylation","Gene Expression","Cancer Type (matching TCGA label)","Drug\nResponse"])
final_data

**HANDLING MISSING VALUES AND DUPLICATES**

In [ ]:
print(final_data.duplicated().sum())

In [ ]:
print(final_data.nunique())

In [ ]:
#DRUG_ID and DRUG_NAME unique values are different
#which suggests that a drug has more than one drug id
drug_pairs=final_data[['DRUG_NAME',"DRUG_ID"]].drop_duplicates()
duplicates=drug_pairs[drug_pairs.duplicated("DRUG_NAME",keep=False)]
duplicates

In [ ]:
#checking if the different id for same drug has the same targets and pathways
#checking with one id at a time
print(final_data.loc[final_data["DRUG_ID"]==1007,["DRUG_NAME","TARGET","TARGET_PATHWAY"]])

It was observed that even if a drug has different drug id, it had the same target and pathway details. so only DRUG_NAME can be used further.

In [ ]:
final_data=final_data.drop(columns=["DRUG_ID"],axis=1)

In [ ]:
final_data.isnull().sum().sort_values(ascending=False)

In [ ]:
#filling missing values in TARGET column based on known targets of the same drug
def target_column(final_data):
  for drug in final_data["DRUG_NAME"].unique():
    drug_mask=final_data["DRUG_NAME"]==drug
    drug_data=final_data[drug_mask] #stores rows for which is true for the specific drug
    known_target=drug_data["TARGET"].dropna().unique() #store non-null target for the drug
    if len(known_target)>0:
      final_data.loc[drug_mask & final_data["TARGET"].isnull(),"TARGET"]=known_target[0]
    else:
      final_data.loc[drug_mask & final_data["TARGET"].isnull(),"TARGET"]="Unknown"
  return final_data

final_data=target_column(final_data)
final_data["TARGET"].isnull().sum()

In [ ]:
#MSI is a cell line property so filling the null values based on cell line
def ms_column(final_data):
  for cell_line in final_data["CELL_LINE_NAME"].unique():
    cell_mask=final_data["CELL_LINE_NAME"]==cell_line
    cell_data=final_data[cell_mask]
    known_msi = cell_data["MSI"].dropna().unique()
    if len(known_msi)>0:
      final_data.loc[cell_mask & final_data["MSI"].isnull(),"MSI"]=known_msi[0]
    else:
      final_data.loc[cell_mask & final_data["MSI"].isnull(),"MSI"]="Unknown"
  return final_data

final_data=ms_column(final_data)
final_data["MSI"].isnull().sum()

In [ ]:
tcga_counts = final_data.groupby("CELL_LINE_NAME")["TCGA_DESC"].nunique() #check whether each cell line is associated with a specific type of cancer
print(tcga_counts[tcga_counts > 1])

In [ ]:
tissue2_counts = final_data.groupby("CELL_LINE_NAME")["GDSC Tissue descriptor 2"].nunique() #check whether each cell line is associated with a specific tissue
print(tissue2_counts[tissue2_counts > 1])

In [ ]:
tissue1_counts = final_data.groupby("CELL_LINE_NAME")["GDSC Tissue descriptor 1"].nunique() #check whether each cell line is associated with a specific tissue
print(tissue1_counts[tissue1_counts > 1])

In [ ]:
#filling missing values for the categorical columns
categorical_col=["GDSC Tissue descriptor 1","GDSC Tissue descriptor 2", "Growth Properties",
"TCGA_DESC"]
def category_fill(final_data):
  for col in categorical_col:
    for cell in final_data["CELL_LINE_NAME"].unique():
      cell_mask=final_data["CELL_LINE_NAME"]==cell
      cell_data=final_data[cell_mask]
      known_value=final_data[cell_mask][col].dropna().unique()
      if len(known_value)>0:
        final_data.loc[cell_mask & final_data[col].isnull(),col]=known_value[0]
      else:
        final_data.loc[cell_mask & final_data[col].isnull(),col]="Unknown"
  return final_data
final_data=category_fill(final_data)

In [ ]:
final_data.isnull().sum()

**MODEL TRAINING AND COMPARISON**

In [ ]:
X=final_data.drop(columns=["LN_IC50"],axis=1)
y=final_data["LN_IC50"]

In [ ]:
category=X.select_dtypes(include="object").columns.tolist()
number=X.select_dtypes(include=["int64","float64"]).columns.tolist()
print("categorical_columns=",category)
print("numerical_columns=",number)

In [ ]:
numeric_transformer=Pipeline(steps=[("scaler",StandardScaler())])
category_transformer=Pipeline(steps=[("encoder",OneHotEncoder(handle_unknown="ignore"))])

In [ ]:
preprocessor=ColumnTransformer(transformers=[("num",numeric_transformer,number),("cat",category_transformer,category)])
#preprocessing step for tree models
preprocessor1=ColumnTransformer(transformers=[("cat",category_transformer,category)])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#defining models
model_linear=Pipeline(steps=[("preprocessor",preprocessor),("model",LinearRegression())])
model_tree=Pipeline(steps=[("preprocessor",preprocessor1),("model",DecisionTreeRegressor(max_depth=250,
                                                                                         min_samples_split=15,min_samples_leaf=10,random_state=42))])
model_random=Pipeline(steps=[("preprocessor",preprocessor1),("model",RandomForestRegressor(n_estimators=100,max_depth=250,min_samples_split=30,
                                                                                           min_samples_leaf=20,max_features=15,random_state=42))])
model_gradient=Pipeline(steps=[("preprocessor",preprocessor1),("model",GradientBoostingRegressor(n_estimators=300,max_depth=30,learning_rate=0.1,
                                                                                                 min_samples_split=20,min_samples_leaf=2,max_features=2,
                                                                                                 random_state=42))])

In [ ]:
results=[]
kf=KFold(n_splits=5,shuffle=True,random_state=42)
models={"LinearRegression":model_linear,"DecisionTree":model_tree,"RandomForest":model_random,"GradientBoosting":model_gradient}
for name,model in models.items():
  model.fit(X_train,y_train)
  pred=model.predict(X_test)
  cv_score=cross_val_score(model,X,y,cv=kf,scoring="r2")
  results.append({"model":name,
                "MSE":mean_squared_error(y_test,pred),
                "MAE":mean_absolute_error(y_test,pred),
                "RMSE":np.sqrt(mean_squared_error(y_test,pred)),
                "R2":r2_score(y_test,pred),"CV mean":cv_score.mean()})
results_df=pd.DataFrame(results)
results_df=results_df.sort_values(by="R2",ascending=False)

In [ ]:
print(results_df)

In [97]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [2, 5, 10],
    "model__max_depth": [10,20,30]
}

gs = GridSearchCV(
    estimator=model_gradient,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best R2:", gs.best_score_)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Best params: {'model__max_depth': 30, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10, 'model__n_estimators': 300}
Best R2: 0.7901750585110262


After training the models and training gradient boost model with hypertuned parameters, linear regression still seemed to perform better based on R2, MSE, CV, RMSE values.

In [ ]:
prediction=model_linear.predict(X_test)

In [ ]:
#actual vs predicted charges
table=pd.DataFrame({'Actual_IC50':y_test.reset_index(drop=True),'Predicted_IC50':prediction})
table['error']=table['Actual_IC50']-table['Predicted_IC50']
table.head(10)

In [ ]:
plt.scatter(y_test, prediction)
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title('Actual vs. Predicted Charges')

**RANKING**

Here, I have tried to rank effective drugs for a particular cancer cell line. (This ranking is not biologically significant)

In [ ]:
kf=KFold(n_splits=5, shuffle=True, random_state=42)
preds=cross_val_predict(model_linear, X, y, cv=kf, n_jobs=-1)

ranking_df=pd.DataFrame({"CELL_LINE_NAME": X["CELL_LINE_NAME"].values,
    "DRUG_NAME": X["DRUG_NAME"].values,
    "Actual_LN_IC50":y.values,
    "Predicted_LN_IC50":preds
})
ranking_df.head()


In [ ]:
ranking_df["Actual_Rank"] = ranking_df.groupby("CELL_LINE_NAME")["Actual_LN_IC50"].rank(method="min", ascending=True)
ranking_df["Predicted_Rank"] = ranking_df.groupby("CELL_LINE_NAME")["Predicted_LN_IC50"].rank(method="min", ascending=True)
ranking_df=ranking_df.sort_values(
    by=["Actual_Rank", "Predicted_Rank"],
    ascending=[True, True])
ranking_df

In [ ]:
min_drug=5
#spearman correlation between actual and predicted values for each cell line

def cell_line_corr(group):
    if len(group)<min_drug:
        return pd.Series({"n_drugs":len(group), "spearman_r":np.nan, "p_value":np.nan})
    r, p=spearmanr(group["Actual_LN_IC50"], group["Predicted_LN_IC50"])
    return pd.Series({"n_drugs": len(group), "spearman_r":r, "p_value":p}) #n_drugs=no.of drugs tested on this cell line

cell_line_spear=ranking_df.groupby("CELL_LINE_NAME").apply(cell_line_corr).dropna()
cell_line_spear=cell_line_spear.sort_values("spearman_r", ascending=False)
cell_line_spear.head(10)

In [ ]:
print("Cell lines evaluated:",len(cell_line_spear))
print("Mean Spearman correlation:",cell_line_spear["spearman_r"].mean())
print("Median Spearman correlation:",cell_line_spear["spearman_r"].median())

plt.hist(cell_line_spear["spearman_r"], bins=30)
plt.xlabel("Spearman rank correlation (per cell line)")
plt.ylabel("Number of cell lines")
plt.title("Distribution of spearman correlation across cell lines")
plt.axvline(0, color="red", linestyle="--")
plt.show()

In [ ]:
# measure of how well predicted top k drugs match the actual top k drugs
def top_k_overlap(group, k=5):
    if len(group)<k:
        return np.nan
    actual_top=set(group.nsmallest(k, "Actual_LN_IC50")["DRUG_NAME"])
    predicted_top=set(group.nsmallest(k, "Predicted_LN_IC50")["DRUG_NAME"])
    return len(actual_top & predicted_top)/k
k=5
overlap_scores=ranking_df.groupby("CELL_LINE_NAME").apply(top_k_overlap, k=k).dropna()
print(f"Mean top-{k} overlap across cell lines:", overlap_scores.mean())
print(f"Median top-{k} overlap across cell lines:", overlap_scores.median())

In [ ]:
#Testing with one cell line
def drug_ranking():
    test_cell_line = input("Enter cell line name: ").strip()

    rank_chart = (ranking_df[ranking_df["CELL_LINE_NAME"].str.lower() == test_cell_line.lower()]
        .sort_values("Actual_Rank")[["DRUG_NAME", "Actual_LN_IC50", "Predicted_LN_IC50","Actual_Rank", "Predicted_Rank"]].reset_index(drop=True))

    if rank_chart.empty:
        print(f"No ranking data found for cell line: {test_cell_line}")
        return

    print(f"Cell line: {test_cell_line}")
    return rank_chart

ranks=drug_ranking()
print(ranks.to_string(index=False))